# Semantic code search with Claude

This notebook builds a semantic search engine over a codebase: instead of
matching keywords, it finds code by *meaning*. Ask "where does the app
authenticate users?" and it retrieves the relevant functions even when the
word "auth" never appears in the source.

The pipeline:

1. **Chunk** the codebase into functions with tree-sitter.
2. **Embed** each function with Voyage AI.
3. **Search** by cosine similarity between the query and the chunks.
4. **Answer** with Claude, grounded in the retrieved functions.
5. **Cache** the embeddings so re-runs do not re-embed the corpus.

This is the same pattern used by developer tools like semantic grep and AI
code assistants.

## Setup

You need two API keys:

- `VOYAGE_API_KEY` (free at [voyageai.com](https://www.voyageai.com)) for embeddings.
- `ANTHROPIC_API_KEY` for the final answer step.

Set them as environment variables before running this notebook.

In [1]:
%pip install -q voyageai tree-sitter tree-sitter-python

import hashlib
import json
import os
from pathlib import Path

import anthropic
import numpy as np
import tree_sitter_python
import voyageai
from tree_sitter import Language, Parser

client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))
vo = voyageai.Client(api_key=os.environ.get("VOYAGE_API_KEY"))

Note: you may need to restart the kernel to use updated packages.


## A tiny example codebase

We search over a small, self-contained codebase stored as a dict of
`filename -> source`. Notice that `session.py` holds the authentication logic,
but the words "auth", "authenticate", and "login" do not appear in it. That
is the point: keyword search would miss it; semantic search will not.

In [2]:
EXAMPLE_CODEBASE = {
    "billing.py": '''def calculate_subtotal(items):
    return sum(item["price"] * item["qty"] for item in items)


def charge_card(amount, card_token):
    return {"status": "ok", "amount": amount, "last4": card_token[-4:]}
''',

    "session.py": '''import hashlib
import secrets


def hash_password(password, salt):
    return hashlib.sha256(f"{salt}:{password}".encode()).hexdigest()


def issue_session_token(user_id):
    return f"{user_id}:{secrets.token_hex(16)}"


def verify_session_token(token):
    user_id, secret = token.split(":")
    return len(secret) == 32


def require_login(handler):
    def wrapped(request):
        token = request.get("session_token")
        if not token or not verify_session_token(token):
            raise PermissionError("missing or invalid session")
        return handler(request)
    return wrapped
''',

    "notify.py": '''def send_welcome_email(email, name):
    print(f"Welcome, {name}, to {email}")


def format_receipt(total, items):
    return " | ".join(f"{i['qty']}x {i['name']}" for i in items)
''',
}

## Chunk the code into functions with tree-sitter

tree-sitter parses source into a concrete syntax tree, so we can extract
function definitions reliably (name plus full source) without regex. Because
tree-sitter is language-agnostic, swapping the grammar for
`tree_sitter_javascript` or `tree_sitter_rust` makes this work for other
languages too.

In [3]:
parser = Parser(Language(tree_sitter_python.language()))


def extract_functions(source: str) -> list[tuple[str, str]]:
    """Return (name, source) for every function defined in `source`."""
    data = source.encode()
    tree = parser.parse(data)
    functions = []

    def walk(node):
        if node.type == "function_definition":
            name_node = node.child_by_field_name("name")
            name = data[name_node.start_byte:name_node.end_byte].decode()
            functions.append((name, data[node.start_byte:node.end_byte].decode()))
        for child in node.children:
            walk(child)

    walk(tree.root_node)
    return functions

## Embed the functions and cache the index

We embed each function with Voyage, normalize the vectors to unit length, and
save the result to disk. A content hash acts as the freshness check: if the
code has not changed, we reload the cache instead of re-embedding, which
saves API calls and money.

In [4]:
MODEL = "voyage-3-lite"

chunks = []
for filename, source in EXAMPLE_CODEBASE.items():
    for name, src in extract_functions(source):
        chunks.append((filename, name, src))

sources = [src for _, _, src in chunks]

INDEX_NPY = Path("semantic_code_index.npy")
INDEX_JSON = Path("semantic_code_index.json")


def embed(texts, input_type):
    """Embed texts with Voyage, returning an (n, d) numpy array."""
    result = vo.embed(texts, model=MODEL, input_type=input_type)
    return np.array(result.embeddings)


def build_index(force=False):
    """Embed the corpus once, cache it, and reuse it if unchanged."""
    digest = hashlib.sha256("\n".join(sources).encode()).hexdigest()
    if (not force and INDEX_NPY.exists() and INDEX_JSON.exists()
            and json.loads(INDEX_JSON.read_text()).get("hash") == digest):
        return np.load(INDEX_NPY)
    vectors = embed(sources, input_type="document")
    vectors = vectors / np.linalg.norm(vectors, axis=1, keepdims=True)
    np.save(INDEX_NPY, vectors)
    INDEX_JSON.write_text(json.dumps({"hash": digest, "n": len(sources)}))
    return vectors


vectors = build_index()

## Search by meaning

Once every vector is unit-length, cosine similarity is just a dot product.
Embedding the query with `input_type="query"` (Voyage treats queries and
documents differently, which improves retrieval) and multiplying it against
the index gives the top-k matches in one line.

In [5]:
def search(query, k=3):
    """Return the top-k functions for a query, as (score, filename, name, src)."""
    q = embed([query], input_type="query")[0]
    q = q / np.linalg.norm(q)
    sims = vectors @ q
    top = np.argsort(sims)[::-1][:k]
    return [(float(sims[i]), *chunks[i]) for i in top]

## Answer with Claude

The final step passes the retrieved functions to Claude as context and asks
it to answer using only that code, naming the functions it relied on.

In [6]:
def answer(query, k=3, hits=None):
    """Answer a question using the top-k matching functions as context."""
    hits = hits if hits is not None else search(query, k=k)
    context = "\n\n".join(
        f"### {filename}:{name}\n```python\n{src}\n```"
        for _, filename, name, src in hits
    )
    message = client.messages.create(
        model="claude-haiku-4-5",
        max_tokens=500,
        system=(
            "You are a code search assistant. Answer the question using ONLY "
            "the provided code snippets, and name the functions you relied on."
        ),
        messages=[{
            "role": "user",
            "content": f"Question: {query}\n\nRelevant code:\n{context}",
        }],
    )
    return message.content[0].text

## Putting it together

One query drives the whole thing: `search` retrieves the matching functions,
then `answer` reuses those results (a single embedding call) to generate the
answer.

In [7]:
query = "how does the app authenticate users?"

hits = search(query)
print("Top matches:")
for score, filename, name, _src in hits:
    print(f"  {score:.3f}  {filename}:{name}")

print()
print(answer(query, hits=hits))

Top matches:
  0.418  session.py:verify_session_token
  0.386  session.py:require_login
  0.381  session.py:wrapped



Based on the provided code, the app authenticates users through a **session token-based system**:

1. **Token Format**: Users are issued a session token in the format `"user_id:secret"` where the secret is a 32-character string.

2. **Verification Process**: The `verify_session_token()` function validates tokens by:
   - Splitting the token on the colon separator
   - Checking that the secret portion is exactly 32 characters long

3. **Request Protection**: The `require_login()` decorator (which returns `wrapped()`) protects endpoints by:
   - Extracting the `session_token` from incoming requests
   - Verifying the token using `verify_session_token()`
   - Raising a `PermissionError` if the token is missing or invalid
   - Allowing the request to proceed only if authentication succeeds

**Functions relied on**: `verify_session_token()`, `require_login()`, and `wrapped()`


## Next steps

- Point it at a real repository by globbing `**/*.py` and chunking each file.
- Swap `tree_sitter_python` for another grammar to support JS, Rust, or Go.
- Add a lexical (BM25) score and blend it with the semantic score.
- Store the chunk sources in the `.json` next to the hash so the index is
  fully self-contained.